# Loading important libraries..

In [3]:
#importing important libraries..
!pip install catboost
!pip install imblearn

import warnings, time
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import gc, os
from sklearn.model_selection import StratifiedKFold, cross_validate, GridSearchCV, train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.feature_selection import mutual_info_classif

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

import xgboost as xgb
import lightgbm as lgb
from catboost import CatBoostClassifier

from sklearn.ensemble import HistGradientBoostingClassifier
from imblearn.ensemble import BalancedRandomForestClassifier
from imblearn.ensemble import EasyEnsembleClassifier


# Display Options
pd.set_option("display.max_columns", None)
pd.set_option("display.max_rows", 100)

# Plot Style
plt.style.use("ggplot")

sns.set_style("whitegrid")
plt.rcParams.update({"figure.dpi": 120, "savefig.dpi": 300, "font.size": 11})
SEED = 42
np.random.seed(SEED)
print("Environment ready")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 7.5 MB/s eta 0:00:00
Environment ready


In [4]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


#Loading the dataset..

In [5]:
df = pd.read_csv("/content/drive/MyDrive/merged_dataset.csv")
print(df.shape)
print(df.head(5).T)

(447915, 77)
                                        0             1             2  \
Flow Duration               214392.000000  2.376792e+06  1.313500e+05   
Total Fwd Packet                 9.000000  9.000000e+00  1.000000e+01   
Total Bwd packets               21.000000  3.000000e+00  3.000000e+00   
Total Length of Fwd Packet     388.000000  7.520000e+02  7.564000e+03   
Total Length of Bwd Packet   24564.000000  0.000000e+00  0.000000e+00   
Fwd Packet Length Max          194.000000  1.880000e+02  1.460000e+03   
Fwd Packet Length Min            0.000000  0.000000e+00  0.000000e+00   
Fwd Packet Length Mean          43.111111  8.355556e+01  7.564000e+02   
Fwd Packet Length Std           85.545959  9.908470e+01  6.904973e+02   
Bwd Packet Length Max         1460.000000  0.000000e+00  0.000000e+00   
Bwd Packet Length Min            0.000000  0.000000e+00  0.000000e+00   
Bwd Packet Length Mean        1169.714286  0.000000e+00  0.000000e+00   
Bwd Packet Length Std          552.156

In [6]:
import shutil
import os

source_path = "/content/drive/MyDrive/merged_dataset.csv"
local_path = "/content/merged_dataset.csv"

# Safely copy the file from Drive to local workspace
if os.path.exists(source_path):
    print("Copying file to local runtime storage for stability...")
    shutil.copyfile(source_path, local_path)
    print("Copy complete.")
else:
    print("Source path not found. Please ensure the file name and path are correct.")

# Read the stable local copy
df = pd.read_csv(local_path)
print("Stable local dataset shape:", df.shape)

Copying file to local runtime storage for stability...
Copy complete.
Stable local dataset shape: (447915, 77)


In [7]:
print(df.shape)
print("------------------------------------------------------------------------")
print(df.info())
print("------------------------------------------------------------------------")
print(df.describe().T)
print("------------------------------------------------------------------------")
print(np.isinf(df.select_dtypes(include=np.number)).sum().sum())
print("------------------------------------------------------------------------")
print(df.isnull().sum().sum())
print("------------------------------------------------------------------------")
print(f"The number of duplicated values in the dataset is : {df.duplicated().sum()}")
print("------------------------------------------------------------------------")
df.columns = df.columns.str.strip()
print(df.columns.to_list())

(447915, 77)
------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 447915 entries, 0 to 447914
Data columns (total 77 columns):
 #   Column                      Non-Null Count   Dtype  
---  ------                      --------------   -----  
 0   Flow Duration               447915 non-null  int64  
 1   Total Fwd Packet            447915 non-null  int64  
 2   Total Bwd packets           447915 non-null  int64  
 3   Total Length of Fwd Packet  447915 non-null  float64
 4   Total Length of Bwd Packet  447915 non-null  float64
 5   Fwd Packet Length Max       447915 non-null  float64
 6   Fwd Packet Length Min       447915 non-null  float64
 7   Fwd Packet Length Mean      447915 non-null  float64
 8   Fwd Packet Length Std       447915 non-null  float64
 9   Bwd Packet Length Max       447915 non-null  float64
 10  Bwd Packet Length Min       447915 non-null  float64
 11  Bwd Packet Length Mean      447915 non-null 

Duplicate handling...

In [8]:
# ==========================================
# DUPLICATE HANDLING
# ==========================================

print("Before duplicate removal:")
print("Rows:", len(df))
l_df = len(df)

df = (
    df
    .drop_duplicates()
    .reset_index(drop=True)
)

print("\nAfter duplicate removal:")
print("Rows:", len(df))

print("\nRows removed:", l_df - len(df))

print(
    "Percentage removed:",
    round((l_df - len(df)) / l_df * 100, 2),
    "%"
)

Before duplicate removal:
Rows: 447915

After duplicate removal:
Rows: 306173

Rows removed: 141742
Percentage removed: 31.64 %


In [9]:
# ==============================
# STEP 1: DATA PREPARATION
# ==============================

# Separate features and target
X = df.drop(columns=["Label"]).copy()
y = df["Label"].copy()

print("Feature shape:", X.shape)
print("Target shape:", y.shape)

print("\nClass distribution:")
print(y.value_counts().sort_index())

print("\nMissing values:", X.isna().sum().sum())
print("Infinite values:", np.isinf(X.select_dtypes(include=np.number)).sum().sum())

Feature shape: (306173, 76)
Target shape: (306173,)

Class distribution:
Label
0    225003
1       385
2       452
3      4412
4     30910
5     26395
6      4496
7     12031
8      1848
9       241
Name: count, dtype: int64

Missing values: 0
Infinite values: 0


In [10]:
# ==========================================
# STEP 2: LEAKAGE-SAFE TRAIN / TEST SPLIT
# ==========================================

from sklearn.model_selection import StratifiedGroupKFold

# Create a group ID for identical feature vectors
groups = pd.util.hash_pandas_object(
    X,
    index=False
)

print("Total samples:", len(X))
print("Unique feature groups:", groups.nunique())

# 5-fold stratified group split
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

train_idx, test_idx = next(
    sgkf.split(X, y, groups=groups)
)

# Create final train and test sets
X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

groups_train = groups.iloc[train_idx]
groups_test = groups.iloc[test_idx]

# ------------------------------------------
# Verify split
# ------------------------------------------

print("\nTrain shape:", X_train.shape)
print("Test shape:", X_test.shape)

print("\nTrain class distribution:")
print(y_train.value_counts(normalize=True).sort_index())

print("\nTest class distribution:")
print(y_test.value_counts(normalize=True).sort_index())

# Check for identical feature groups across train/test
overlap = set(groups_train).intersection(set(groups_test))

print("\nFeature-group overlap:", len(overlap))

Total samples: 306173
Unique feature groups: 305602

Train shape: (244946, 76)
Test shape: (61227, 76)

Train class distribution:
Label
0    0.735060
1    0.001229
2    0.001449
3    0.014485
4    0.100892
5    0.085970
6    0.014722
7    0.039364
8    0.006014
9    0.000817
Name: proportion, dtype: float64

Test class distribution:
Label
0    0.734202
1    0.001372
2    0.001584
3    0.014111
4    0.101214
5    0.087167
6    0.014536
7    0.039019
8    0.006125
9    0.000670
Name: proportion, dtype: float64

Feature-group overlap: 0


In [11]:
# ==========================================
# STEP 3: DEVELOPMENT SUBSET
# ==========================================

from sklearn.model_selection import train_test_split

# Create development subset from TRAINING DATA ONLY
X_dev, _, y_dev, _ = train_test_split(
    X_train,
    y_train,
    train_size=120_000,
    stratify=y_train,
    random_state=42
)

print("Development feature shape:", X_dev.shape)
print("Development target shape:", y_dev.shape)

print("\nDevelopment class distribution:")
print(y_dev.value_counts(normalize=True).sort_index())

print("\nFinal test set remains:")
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

Development feature shape: (120000, 76)
Development target shape: (120000,)

Development class distribution:
Label
0    0.735058
1    0.001225
2    0.001450
3    0.014483
4    0.100892
5    0.085967
6    0.014725
7    0.039367
8    0.006017
9    0.000817
Name: proportion, dtype: float64

Final test set remains:
X_test: (61227, 76)
y_test: (61227,)


In [12]:
# ============================================================
# STEP 4: PREPROCESSING & MODEL PIPELINE ARCHITECTURE
# ============================================================

# ------------------------------------------------------------
# 1. Imports
# ------------------------------------------------------------

from sklearn.pipeline import Pipeline
from imblearn.pipeline import Pipeline as ImbPipeline

from sklearn.compose import ColumnTransformer

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import (
    StandardScaler,
    OneHotEncoder
)

from imblearn.over_sampling import SMOTE


# ------------------------------------------------------------
# 2. Reproducibility
# ------------------------------------------------------------

SEED = 42


# ------------------------------------------------------------
# 3. Identify Feature Types
# ------------------------------------------------------------

num_cols = X_train.select_dtypes(
    include=["number"]
).columns.tolist()

cat_cols = X_train.select_dtypes(
    exclude=["number"]
).columns.tolist()


print("=" * 55)
print("FEATURE TYPE SUMMARY")
print("=" * 55)

print(f"Numerical Features   : {len(num_cols)}")
print(f"Categorical Features : {len(cat_cols)}")


# ------------------------------------------------------------
# 4. Numerical Preprocessing Pipeline
# ------------------------------------------------------------

def create_numeric_pipeline(scale=False):
    """
    Create preprocessing pipeline for numerical features.

    Parameters
    ----------
    scale : bool, default=False
        Whether to apply StandardScaler.

    Returns
    -------
    Pipeline
        Numerical preprocessing pipeline.
    """

    steps = [
        (
            "imputer",
            SimpleImputer(strategy="median")
        )
    ]

    if scale:
        steps.append(
            (
                "scaler",
                StandardScaler()
            )
        )

    return Pipeline(steps)


# ------------------------------------------------------------
# 5. Categorical Preprocessing Pipeline
# ------------------------------------------------------------

def create_categorical_pipeline():
    """
    Create preprocessing pipeline for categorical features.

    Returns
    -------
    Pipeline
        Categorical preprocessing pipeline.
    """

    return Pipeline([
        (
            "imputer",
            SimpleImputer(strategy="most_frequent")
        ),

        (
            "onehot",
            OneHotEncoder(
                handle_unknown="ignore"
            )
        )
    ])


# ------------------------------------------------------------
# 6. Create Column-Wise Preprocessor
# ------------------------------------------------------------

def create_preprocessor(
    num_cols,
    cat_cols,
    scale=False
):
    """
    Create a column-wise preprocessing transformer.

    Numerical features:
        Median imputation
        Optional standardization

    Categorical features:
        Most-frequent imputation
        One-hot encoding

    Parameters
    ----------
    num_cols : list
        Numerical feature names.

    cat_cols : list
        Categorical feature names.

    scale : bool, default=False
        Whether numerical features should be standardized.

    Returns
    -------
    ColumnTransformer
        Combined preprocessing transformer.
    """

    transformers = []

    # Numerical features
    if num_cols:

        transformers.append(
            (
                "num",
                create_numeric_pipeline(
                    scale=scale
                ),
                num_cols
            )
        )

    # Categorical features
    if cat_cols:

        transformers.append(
            (
                "cat",
                create_categorical_pipeline(),
                cat_cols
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


# ------------------------------------------------------------
# 7. General Model Pipeline Builder
# ------------------------------------------------------------

def build_pipeline(
    model,
    num_cols,
    cat_cols,
    scale=False,
    sampler=None
):
    """
    Build a complete leakage-safe ML pipeline.

    Pipeline structure:

        Raw Data
            ↓
        Preprocessor
            ↓
        Optional Sampler
            ↓
        Model

    Parameters
    ----------
    model : estimator
        Machine learning model.

    num_cols : list
        Numerical feature names.

    cat_cols : list
        Categorical feature names.

    scale : bool, default=False
        Whether numerical features should be standardized.

    sampler : sampler or None, default=None
        Optional imbalance-handling sampler such as SMOTE.

    Returns
    -------
    ImbPipeline
        Complete preprocessing + sampling + model pipeline.
    """

    preprocessor = create_preprocessor(
        num_cols=num_cols,
        cat_cols=cat_cols,
        scale=scale
    )

    steps = [
        (
            "preprocessor",
            preprocessor
        )
    ]

    # Optional imbalance-handling step
    if sampler is not None:

        steps.append(
            (
                "sampler",
                sampler
            )
        )

    # Final estimator
    steps.append(
        (
            "model",
            model
        )
    )

    return ImbPipeline(steps)


# ------------------------------------------------------------
# 8. Architecture Verification
# ------------------------------------------------------------

print("\n" + "=" * 55)
print("PIPELINE ARCHITECTURE READY")
print("=" * 55)

print("Preprocessing : ColumnTransformer")
print("Numerical     : Median Imputation")
print("Scaling       : Model-dependent")
print("Categorical   : Imputation + One-Hot Encoding")
print("Sampling      : Optional")
print("Model         : Configurable")
print("Leakage Safe  : Yes")

FEATURE TYPE SUMMARY
Numerical Features   : 76
Categorical Features : 0

PIPELINE ARCHITECTURE READY
Preprocessing : ColumnTransformer
Numerical     : Median Imputation
Scaling       : Model-dependent
Categorical   : Imputation + One-Hot Encoding
Sampling      : Optional
Model         : Configurable
Leakage Safe  : Yes


In [13]:
# ============================================================
# STEP 5: MODEL REGISTRY
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    HistGradientBoostingClassifier
)


# ------------------------------------------------------------
# 1. Model Registry
# ------------------------------------------------------------
 ## Models which doesn't use smote..

MODEL_REGISTRY = {

    # --------------------------------------------------------
    # Logistic Regression
    # --------------------------------------------------------
    "Logistic Regression": {
        "model": LogisticRegression(
            max_iter=1000,
            random_state=SEED
        ),
        "scale": True,
        "sampler": None
    },


    # --------------------------------------------------------
    # Decision Tree
    # --------------------------------------------------------
    "Decision Tree": {
        "model": DecisionTreeClassifier(
            random_state=SEED
        ),
        "scale": False,
        "sampler": None
    },


    # --------------------------------------------------------
    # Random Forest
    # --------------------------------------------------------
    "Random Forest": {
        "model": RandomForestClassifier(
            n_estimators=200,
            random_state=SEED,
            n_jobs=-1
        ),
        "scale": False,
        "sampler": None
    },


    # --------------------------------------------------------
    # Histogram Gradient Boosting
    # --------------------------------------------------------
    "HistGradientBoosting": {
        "model": HistGradientBoostingClassifier(
            random_state=SEED
        ),
        "scale": False,
        "sampler": None
    }
}


# ------------------------------------------------------------
# 2. Build Model Pipelines
# ------------------------------------------------------------

PIPELINES = {}

for model_name, config in MODEL_REGISTRY.items():

    PIPELINES[model_name] = build_pipeline(
        model=config["model"],
        num_cols=num_cols,
        cat_cols=cat_cols,
        scale=config["scale"],
        sampler=config["sampler"]
    )


# ------------------------------------------------------------
# 3. Registry Summary
# ------------------------------------------------------------

print("=" * 60)
print("MODEL REGISTRY")
print("=" * 60)

for model_name, config in MODEL_REGISTRY.items():

    print(
        f"{model_name:<25} | "
        f"Scaling: {str(config['scale']):<5} | "
        f"Sampler: "
        f"{type(config['sampler']).__name__ if config['sampler'] else 'None'}"
    )

print("=" * 60)
print(f"Total models registered: {len(PIPELINES)}")

MODEL REGISTRY
Logistic Regression       | Scaling: True  | Sampler: None
Decision Tree             | Scaling: False | Sampler: None
Random Forest             | Scaling: False | Sampler: None
HistGradientBoosting      | Scaling: False | Sampler: None
Total models registered: 4


In [14]:
# ============================================================
# STEP 6: REUSABLE EVALUATION ENGINE
# ============================================================

import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay
)


# ------------------------------------------------------------
# 1. Model Evaluation Function
# ------------------------------------------------------------

def evaluate_model(
    model_name,
    pipeline,
    X_train_data,
    y_train_data,
    X_eval_data,
    y_eval_data,
    show_report=True,
    show_confusion_matrix=True
):
    """
    Train and evaluate a model using a unified evaluation protocol.
    """

    print("\n" + "=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    # --------------------------------------------------------
    # Training
    # --------------------------------------------------------

    start_train = time.perf_counter()

    pipeline.fit(
        X_train_data,
        y_train_data
    )

    train_time = time.perf_counter() - start_train


    # --------------------------------------------------------
    # Prediction
    # --------------------------------------------------------

    start_predict = time.perf_counter()

    y_pred = pipeline.predict(
        X_eval_data
    )

    prediction_time = time.perf_counter() - start_predict


    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_eval_data,
        y_pred
    )

    macro_precision = precision_score(
        y_eval_data,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_recall = recall_score(
        y_eval_data,
        y_pred,
        average="macro",
        zero_division=0
    )

    macro_f1 = f1_score(
        y_eval_data,
        y_pred,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        y_eval_data,
        y_pred,
        average="weighted",
        zero_division=0
    )


    # --------------------------------------------------------
    # Summary
    # --------------------------------------------------------

    print(f"\nTraining Time   : {train_time:.2f} seconds")
    print(f"Prediction Time : {prediction_time:.2f} seconds")

    print(f"\nAccuracy        : {accuracy:.4f}")
    print(f"Macro Precision : {macro_precision:.4f}")
    print(f"Macro Recall    : {macro_recall:.4f}")
    print(f"Macro F1        : {macro_f1:.4f}")
    print(f"Weighted F1     : {weighted_f1:.4f}")


    # --------------------------------------------------------
    # Classification Report
    # --------------------------------------------------------

    if show_report:

        print("\nClassification Report:")
        print(
            classification_report(
                y_eval_data,
                y_pred,
                zero_division=0
            )
        )


    # --------------------------------------------------------
    # Confusion Matrix
    # --------------------------------------------------------

    if show_confusion_matrix:

        cm = confusion_matrix(
            y_eval_data,
            y_pred
        )

        plt.figure(figsize=(10, 8))

        disp = ConfusionMatrixDisplay(
            confusion_matrix=cm
        )

        disp.plot(
            xticks_rotation="vertical"
        )

        plt.title(
            f"Confusion Matrix — {model_name}"
        )

        plt.tight_layout()
        plt.show()


    # --------------------------------------------------------
    # Return Results
    # --------------------------------------------------------

    return {
        "Model": model_name,
        "Accuracy": accuracy,
        "Macro Precision": macro_precision,
        "Macro Recall": macro_recall,
        "Macro F1": macro_f1,
        "Weighted F1": weighted_f1,
        "Training Time (s)": train_time,
        "Prediction Time (s)": prediction_time,
        "y_true": y_eval_data,
        "y_pred": y_pred,
        "pipeline": pipeline
    }

In [15]:
# ============================================================
# 2. Benchmark Runner
# ============================================================

def run_benchmark(
    pipelines,
    X_train_data,
    y_train_data,
    X_eval_data,
    y_eval_data,
    show_report=False,
    show_confusion_matrix=False
):
    """
    Run all registered models using the same evaluation protocol.
    """

    results = []

    for model_name, pipeline in pipelines.items():

        result = evaluate_model(
            model_name=model_name,
            pipeline=pipeline,
            X_train_data=X_train_data,
            y_train_data=y_train_data,
            X_eval_data=X_eval_data,
            y_eval_data=y_eval_data,
            show_report=show_report,
            show_confusion_matrix=show_confusion_matrix
        )

        results.append(result)


    # --------------------------------------------------------
    # Create Benchmark Table
    # --------------------------------------------------------

    results_df = pd.DataFrame([
        {
            key: value
            for key, value in result.items()
            if key not in [
                "y_true",
                "y_pred",
                "pipeline"
            ]
        }
        for result in results
    ])

    results_df = results_df.sort_values(
        by="Macro F1",
        ascending=False
    ).reset_index(drop=True)

    return results, results_df

In [16]:
# ============================================================
# 3. Development Benchmark
# ============================================================

benchmark_results, benchmark_df = run_benchmark(
    pipelines=PIPELINES,

    X_train_data=X_dev,
    y_train_data=y_dev,

    X_eval_data=X_dev,
    y_eval_data=y_dev,

    show_report=False,
    show_confusion_matrix=False
)


MODEL: Logistic Regression

Training Time   : 113.27 seconds
Prediction Time : 0.48 seconds

Accuracy        : 0.8640
Macro Precision : 0.5441
Macro Recall    : 0.3171
Macro F1        : 0.3239
Weighted F1     : 0.8612

MODEL: Decision Tree

Training Time   : 10.76 seconds
Prediction Time : 0.26 seconds

Accuracy        : 0.9991
Macro Precision : 0.9986
Macro Recall    : 0.9908
Macro F1        : 0.9946
Weighted F1     : 0.9991

MODEL: Random Forest

Training Time   : 109.67 seconds
Prediction Time : 3.66 seconds

Accuracy        : 0.9991
Macro Precision : 0.9959
Macro Recall    : 0.9932
Macro F1        : 0.9945
Weighted F1     : 0.9991

MODEL: HistGradientBoosting

Training Time   : 11.15 seconds
Prediction Time : 1.39 seconds

Accuracy        : 0.9026
Macro Precision : 0.5716
Macro Recall    : 0.5432
Macro F1        : 0.5248
Weighted F1     : 0.9020


In [17]:
# ============================================================
# STEP 6: GROUP-AWARE DEVELOPMENT VALIDATION SPLIT
# ============================================================

from sklearn.model_selection import StratifiedGroupKFold


# ------------------------------------------------------------
# 1. Create feature groups for the development set
# ------------------------------------------------------------

groups_dev = pd.util.hash_pandas_object(
    X_dev,
    index=False
)


print("=" * 60)
print("DEVELOPMENT GROUP INFORMATION")
print("=" * 60)

print(f"Development samples : {len(X_dev):,}")
print(f"Unique feature groups: {groups_dev.nunique():,}")
print(
    f"Duplicate feature groups: "
    f"{len(groups_dev) - groups_dev.nunique():,}"
)

DEVELOPMENT GROUP INFORMATION
Development samples : 120,000
Unique feature groups: 119,889
Duplicate feature groups: 111


In [18]:
# ------------------------------------------------------------
# 2. Group-aware stratified split
# ------------------------------------------------------------

sgkf_dev = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)


train_idx, val_idx = next(
    sgkf_dev.split(
        X_dev,
        y_dev,
        groups=groups_dev
    )
)


# ------------------------------------------------------------
# 3. Create development train/validation sets
# ------------------------------------------------------------

X_dev_train = X_dev.iloc[train_idx].copy()
X_dev_val = X_dev.iloc[val_idx].copy()

y_dev_train = y_dev.iloc[train_idx].copy()
y_dev_val = y_dev.iloc[val_idx].copy()

groups_dev_train = groups_dev.iloc[train_idx]
groups_dev_val = groups_dev.iloc[val_idx]

In [19]:
# ============================================================
# 4. VALIDATION SPLIT VERIFICATION
# ============================================================

print("\n" + "=" * 60)
print("DEVELOPMENT SPLIT")
print("=" * 60)

print(f"Training samples   : {len(X_dev_train):,}")
print(f"Validation samples : {len(X_dev_val):,}")


print("\nClass distribution:")
print(
    pd.DataFrame({
        "Train": y_dev_train.value_counts().sort_index(),
        "Validation": y_dev_val.value_counts().sort_index()
    })
)


DEVELOPMENT SPLIT
Training samples   : 96,002
Validation samples : 23,998

Class distribution:
       Train  Validation
Label                   
0      70550       17657
1        108          39
2        132          42
3       1420         318
4       9698        2409
5       8242        2074
6       1425         342
7       3780         944
8        570         152
9         77          21


In [20]:
# ------------------------------------------------------------
# 5. Verify zero feature-group overlap
# ------------------------------------------------------------

group_overlap = set(
    groups_dev_train
).intersection(
    set(groups_dev_val)
)


print("\n" + "=" * 60)
print("GROUP LEAKAGE CHECK")
print("=" * 60)

print(
    f"Overlapping feature groups : {len(group_overlap)}"
)


GROUP LEAKAGE CHECK
Overlapping feature groups : 0


In [21]:
# ============================================================
# STEP 6.4: PROPER DEVELOPMENT BENCHMARK
# ============================================================

dev_results, dev_results_df = run_benchmark(
    pipelines=PIPELINES,

    X_train_data=X_dev_train,
    y_train_data=y_dev_train,

    X_eval_data=X_dev_val,
    y_eval_data=y_dev_val,

    show_report=False,
    show_confusion_matrix=False
)


MODEL: Logistic Regression

Training Time   : 96.13 seconds
Prediction Time : 0.06 seconds

Accuracy        : 0.8673
Macro Precision : 0.5194
Macro Recall    : 0.3179
Macro F1        : 0.3222
Weighted F1     : 0.8642

MODEL: Decision Tree

Training Time   : 10.17 seconds
Prediction Time : 0.05 seconds

Accuracy        : 0.8816
Macro Precision : 0.5270
Macro Recall    : 0.5064
Macro F1        : 0.5152
Weighted F1     : 0.8812

MODEL: Random Forest

Training Time   : 90.81 seconds
Prediction Time : 1.10 seconds

Accuracy        : 0.9037
Macro Precision : 0.6280
Macro Recall    : 0.4986
Macro F1        : 0.5383
Weighted F1     : 0.9018

MODEL: HistGradientBoosting

Training Time   : 14.66 seconds
Prediction Time : 0.54 seconds

Accuracy        : 0.8964
Macro Precision : 0.5043
Macro Recall    : 0.4528
Macro F1        : 0.4531
Weighted F1     : 0.8964


In [22]:


# ------------------------------------------------------------
# Sorted Model Comparison
# ------------------------------------------------------------

dev_results_df[
    [
        "Model",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1",
        "Training Time (s)",
        "Prediction Time (s)"
    ]
]

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Training Time (s),Prediction Time (s)
0,Random Forest,0.903659,0.627959,0.498618,0.538330,0.901804,90.806255,1.097115
1,Decision Tree,0.881573,0.526958,0.506445,0.515201,0.881214,10.169917,0.051175
2,HistGradientBoosting,0.896366,0.504336,0.452758,0.453139,0.896364,14.658423,0.538044
3,Logistic Regression,0.867281,0.519392,0.317857,0.322221,0.864208,96.134933,0.061533


In [23]:
# ============================================================
# STEP 6.5: PER-CLASS BASELINE ANALYSIS
# ============================================================

label_names = {
    0: "Benign",
    1: "Analysis",
    2: "Backdoor",
    3: "DoS",
    4: "Exploits",
    5: "Fuzzers",
    6: "Generic",
    7: "Reconnaissance",
    8: "Shellcode",
    9: "Worms"
}


for result in dev_results:

    print("\n" + "=" * 80)
    print(f"MODEL: {result['Model']}")
    print("=" * 80)

    report = classification_report(
        result["y_true"],
        result["y_pred"],
        labels=list(label_names.keys()),
        target_names=list(label_names.values()),
        output_dict=True,
        zero_division=0
    )

    class_report = pd.DataFrame(report).T

    class_report = class_report.loc[
        list(label_names.values())
    ]

    print(
        class_report[
            [
                "precision",
                "recall",
                "f1-score",
                "support"
            ]
        ].round(4)
    )


MODEL: Logistic Regression
                precision  recall  f1-score  support
Benign             0.9951  0.9626    0.9786  17657.0
Analysis           0.0000  0.0000    0.0000     39.0
Backdoor           1.0000  0.0238    0.0465     42.0
DoS                0.4286  0.0094    0.0185    318.0
Exploits           0.7000  0.6480    0.6730   2409.0
Fuzzers            0.4462  0.8616    0.5879   2074.0
Generic            0.6439  0.2485    0.3586    342.0
Reconnaissance     0.7135  0.3983    0.5112    944.0
Shellcode          0.2667  0.0263    0.0479    152.0
Worms              0.0000  0.0000    0.0000     21.0

MODEL: Decision Tree
                precision  recall  f1-score  support
Benign             0.9737  0.9767    0.9752  17657.0
Analysis           0.2821  0.2821    0.2821     39.0
Backdoor           0.5625  0.4286    0.4865     42.0
DoS                0.2714  0.2987    0.2844    318.0
Exploits           0.7000  0.6924    0.6962   2409.0
Fuzzers            0.6428  0.6369    0.6399   207

In [24]:
# ============================================================
# STEP 7 — CLASS-WEIGHTED MODEL REGISTRY
# ============================================================

WEIGHTED_MODEL_REGISTRY = {

    "Logistic Regression (Balanced)": {
        "model": LogisticRegression(
            max_iter=1000,
            class_weight="balanced",
            random_state=SEED
        ),
        "scale": True,
        "sampler": None
    },

    "Decision Tree (Balanced)": {
        "model": DecisionTreeClassifier(
            class_weight="balanced",
            random_state=SEED
        ),
        "scale": False,
        "sampler": None
    },

    "Random Forest (Balanced)": {
        "model": RandomForestClassifier(
            n_estimators=200,
            class_weight="balanced",
            random_state=SEED,
            n_jobs=-1
        ),
        "scale": False,
        "sampler": None

    }
}

In [25]:
# ============================================================
# BUILD WEIGHTED PIPELINES
# ============================================================

WEIGHTED_PIPELINES = {}

for model_name, config in WEIGHTED_MODEL_REGISTRY.items():

    WEIGHTED_PIPELINES[model_name] = build_pipeline(
        model=config["model"],
        num_cols=num_cols,
        cat_cols=cat_cols,
        scale=config["scale"],
        sampler=config["sampler"]
    )

In [26]:
# ============================================================
# RUN CLASS-WEIGHTED BENCHMARK
# ============================================================

weighted_results, weighted_results_df = run_benchmark(
    pipelines=WEIGHTED_PIPELINES,
    X_train_data=X_dev_train,
    y_train_data=y_dev_train,
    X_eval_data=X_dev_val,
    y_eval_data=y_dev_val,
    show_report=False,
    show_confusion_matrix=False
)

weighted_results_df


MODEL: Logistic Regression (Balanced)

Training Time   : 225.51 seconds
Prediction Time : 0.08 seconds

Accuracy        : 0.8192
Macro Precision : 0.3683
Macro Recall    : 0.5928
Macro F1        : 0.3633
Weighted F1     : 0.8423

MODEL: Decision Tree (Balanced)

Training Time   : 6.47 seconds
Prediction Time : 0.05 seconds

Accuracy        : 0.8791
Macro Precision : 0.5097
Macro Recall    : 0.5007
Macro F1        : 0.5043
Weighted F1     : 0.8792

MODEL: Random Forest (Balanced)

Training Time   : 68.81 seconds
Prediction Time : 1.60 seconds

Accuracy        : 0.9047
Macro Precision : 0.6569
Macro Recall    : 0.5081
Macro F1        : 0.5524
Weighted F1     : 0.9029


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Training Time (s),Prediction Time (s)
0,Random Forest (Balanced),0.904742,0.656933,0.508097,0.552423,0.902885,68.814152,1.602659
1,Decision Tree (Balanced),0.879073,0.509693,0.500691,0.504286,0.879218,6.468021,0.052161
2,Logistic Regression (Balanced),0.819193,0.368254,0.592768,0.363281,0.842274,225.506780,0.082606


In [27]:
# ============================================================
# BASELINE vs WEIGHTED COMPARISON
# ============================================================

baseline_summary = dev_results_df[
    [
        "Model",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1"
    ]
].copy()

weighted_summary = weighted_results_df[
    [
        "Model",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1"
    ]
].copy()

comparison_df = pd.concat(
    [baseline_summary, weighted_summary],
    ignore_index=True
)

comparison_df.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,Random Forest (Balanced),0.904742,0.656933,0.508097,0.552423,0.902885
1,Random Forest,0.903659,0.627959,0.498618,0.538330,0.901804
2,Decision Tree,0.881573,0.526958,0.506445,0.515201,0.881214
3,Decision Tree (Balanced),0.879073,0.509693,0.500691,0.504286,0.879218
4,HistGradientBoosting,0.896366,0.504336,0.452758,0.453139,0.896364
5,Logistic Regression (Balanced),0.819193,0.368254,0.592768,0.363281,0.842274
6,Logistic Regression,0.867281,0.519392,0.317857,0.322221,0.864208


In [28]:
# ============================================================
# PER-CLASS RESULTS FOR WEIGHTED MODELS
# ============================================================

for result in weighted_results:

    print("\n" + "=" * 70)
    print(result["Model"])
    print("=" * 70)

    print(
        classification_report(
            result["y_true"],
            result["y_pred"],
            zero_division=0
        )
    )


Logistic Regression (Balanced)
              precision    recall  f1-score   support

           0       1.00      0.96      0.98     17657
           1       0.13      1.00      0.23        39
           2       0.09      0.67      0.16        42
           3       0.16      0.10      0.12       318
           4       0.86      0.39      0.54      2409
           5       0.55      0.38      0.45      2074
           6       0.35      0.55      0.43       342
           7       0.41      0.66      0.51       944
           8       0.12      0.55      0.19       152
           9       0.01      0.67      0.03        21

    accuracy                           0.82     23998
   macro avg       0.37      0.59      0.36     23998
weighted avg       0.89      0.82      0.84     23998


Decision Tree (Balanced)
              precision    recall  f1-score   support

           0       0.97      0.97      0.97     17657
           1       0.36      0.31      0.33        39
           2       0

In [29]:
# ============================================================
# STEP 8 — SMOTE MODEL REGISTRY
# ============================================================

SMOTE_MODEL_REGISTRY = {

    "Logistic Regression (SMOTE)": {
        "model": LogisticRegression(
            max_iter=1000,
            random_state=SEED
        ),
        "scale": True,
        "sampler": SMOTE(
            random_state=SEED,
            k_neighbors=3
        )
    },

    "Decision Tree (SMOTE)": {
        "model": DecisionTreeClassifier(
            random_state=SEED
        ),
        "scale": False,
        "sampler": SMOTE(
            random_state=SEED,
            k_neighbors=3
        )
    },

    "Random Forest (SMOTE)": {
        "model": RandomForestClassifier(
            n_estimators=200,
            random_state=SEED,
            n_jobs=-1
        ),
        "scale": False,
        "sampler": SMOTE(
            random_state=SEED,
            k_neighbors=3
        )
    }
}

In [34]:
# ============================================================
# BUILD SMOTE PIPELINES
# ============================================================

SMOTE_PIPELINES = {}

for model_name, config in SMOTE_MODEL_REGISTRY.items():

    SMOTE_PIPELINES[model_name] = build_pipeline(
        model=config["model"],
        num_cols=num_cols,
        cat_cols=cat_cols,
        scale=config["scale"],
        sampler=config["sampler"]
    )

In [35]:
# ============================================================
# RUN SMOTE BENCHMARK
# ============================================================

smote_results, smote_results_df = run_benchmark(
    pipelines=SMOTE_PIPELINES,
    X_train_data=X_dev_train,
    y_train_data=y_dev_train,
    X_eval_data=X_dev_val,
    y_eval_data=y_dev_val,
    show_report=False,
    show_confusion_matrix=False
)

smote_results_df


MODEL: Logistic Regression (SMOTE)

Training Time   : 841.34 seconds
Prediction Time : 0.15 seconds

Accuracy        : 0.8264
Macro Precision : 0.3765
Macro Recall    : 0.5992
Macro F1        : 0.3770
Weighted F1     : 0.8490

MODEL: Decision Tree (SMOTE)

Training Time   : 154.19 seconds
Prediction Time : 0.08 seconds

Accuracy        : 0.8723
Macro Precision : 0.4392
Macro Recall    : 0.4725
Macro F1        : 0.4509
Weighted F1     : 0.8760

MODEL: Random Forest (SMOTE)

Training Time   : 1648.12 seconds
Prediction Time : 1.39 seconds

Accuracy        : 0.8911
Macro Precision : 0.5052
Macro Recall    : 0.5230
Macro F1        : 0.5102
Weighted F1     : 0.8948


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Training Time (s),Prediction Time (s)
0,Random Forest (SMOTE),0.891074,0.505172,0.523031,0.510238,0.894772,1648.116062,1.387479
1,Decision Tree (SMOTE),0.872281,0.439174,0.472484,0.450857,0.875964,154.188967,0.082578
2,Logistic Regression (SMOTE),0.826444,0.376514,0.599208,0.377040,0.849025,841.335633,0.149096


In [36]:
# ============================================================
# COMPLETE IMBALANCE STRATEGY COMPARISON
# ============================================================

baseline_summary = dev_results_df[
    [
        "Model",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1"
    ]
].copy()

weighted_summary = weighted_results_df[
    [
        "Model",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1"
    ]
].copy()

smote_summary = smote_results_df[
    [
        "Model",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1"
    ]
].copy()

imbalance_comparison_df = pd.concat(
    [
        baseline_summary,
        weighted_summary,
        smote_summary
    ],
    ignore_index=True
)

imbalance_comparison_df = imbalance_comparison_df.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

imbalance_comparison_df

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1
0,Random Forest (Balanced),0.904742,0.656933,0.508097,0.552423,0.902885
1,Random Forest,0.903659,0.627959,0.498618,0.538330,0.901804
2,Decision Tree,0.881573,0.526958,0.506445,0.515201,0.881214
3,Random Forest (SMOTE),0.891074,0.505172,0.523031,0.510238,0.894772
4,Decision Tree (Balanced),0.879073,0.509693,0.500691,0.504286,0.879218
5,HistGradientBoosting,0.896366,0.504336,0.452758,0.453139,0.896364
6,Decision Tree (SMOTE),0.872281,0.439174,0.472484,0.450857,0.875964
7,Logistic Regression (SMOTE),0.826444,0.376514,0.599208,0.377040,0.849025
8,Logistic Regression (Balanced),0.819193,0.368254,0.592768,0.363281,0.842274
9,Logistic Regression,0.867281,0.519392,0.317857,0.322221,0.864208


In [37]:
# ============================================================
# PER-CLASS RESULTS — SMOTE
# ============================================================

for result in smote_results:

    print("\n" + "=" * 70)
    print(result["Model"])
    print("=" * 70)

    print(
        classification_report(
            result["y_true"],
            result["y_pred"],
            zero_division=0
        )
    )


Logistic Regression (SMOTE)
              precision    recall  f1-score   support

           0       1.00      0.96      0.98     17657
           1       0.15      1.00      0.25        39
           2       0.10      0.67      0.18        42
           3       0.17      0.11      0.13       318
           4       0.87      0.40      0.55      2409
           5       0.55      0.46      0.50      2074
           6       0.34      0.56      0.42       342
           7       0.45      0.63      0.53       944
           8       0.13      0.55      0.21       152
           9       0.01      0.67      0.03        21

    accuracy                           0.83     23998
   macro avg       0.38      0.60      0.38     23998
weighted avg       0.90      0.83      0.85     23998


Decision Tree (SMOTE)
              precision    recall  f1-score   support

           0       0.98      0.97      0.98     17657
           1       0.24      0.18      0.21        39
           2       0.39   

In [44]:
# ============================================================
# STEP 9 — ADVANCED MODELS
# IMPORTS
# ============================================================

from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

from imblearn.ensemble import BalancedRandomForestClassifier

# ============================================================
# ADVANCED MODEL REGISTRY
# ============================================================

ADVANCED_MODEL_REGISTRY = {

    "XGBoost": {
        "model": XGBClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            objective="multi:softmax",
            eval_metric="mlogloss",
            random_state=SEED,
            n_jobs=-1
        ),
        "scale": False,
        "sampler": None
    },

    "LightGBM": {
        "model": LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            max_depth=-1,
            num_leaves=31,
            subsample=0.8,
            colsample_bytree=0.8,
            random_state=SEED,
            n_jobs=-1,
            verbosity=-1
        ),
        "scale": False,
        "sampler": None
    },

    "CatBoost": {
        "model": CatBoostClassifier(
            iterations=300,
            learning_rate=0.05,
            depth=6,
            loss_function="MultiClass",
            random_seed=SEED,
            verbose=False,
            thread_count=-1
        ),
        "scale": False,
        "sampler": None
    },

    "Balanced Random Forest": {
        "model": BalancedRandomForestClassifier(
            n_estimators=200,
            random_state=SEED,
            n_jobs=-1
        ),
        "scale": False,
        "sampler": None
    }
}



In [45]:
# ============================================================
# BUILD ADVANCED PIPELINES
# ============================================================

ADVANCED_PIPELINES = {}

for model_name, config in ADVANCED_MODEL_REGISTRY.items():

    ADVANCED_PIPELINES[model_name] = build_pipeline(
        model=config["model"],
        num_cols=num_cols,
        cat_cols=cat_cols,
        scale=config["scale"],
        sampler=config["sampler"]
    )




In [46]:
# ============================================================
# STEP 9.3 — ADVANCED MODEL BENCHMARK
# ============================================================

advanced_results, advanced_results_df = run_benchmark(
    pipelines=ADVANCED_PIPELINES,
    X_train_data=X_dev_train,
    y_train_data=y_dev_train,
    X_eval_data=X_dev_val,
    y_eval_data=y_dev_val,
    show_report=False,
    show_confusion_matrix=False
)

advanced_results_df


MODEL: XGBoost

Training Time   : 129.45 seconds
Prediction Time : 1.64 seconds

Accuracy        : 0.9083
Macro Precision : 0.7130
Macro Recall    : 0.4863
Macro F1        : 0.5385
Weighted F1     : 0.9056

MODEL: LightGBM

Training Time   : 87.24 seconds
Prediction Time : 4.65 seconds

Accuracy        : 0.8947
Macro Precision : 0.4878
Macro Recall    : 0.4610
Macro F1        : 0.4618
Weighted F1     : 0.8941

MODEL: CatBoost

Training Time   : 295.94 seconds
Prediction Time : 0.20 seconds

Accuracy        : 0.9001
Macro Precision : 0.5669
Macro Recall    : 0.3871
Macro F1        : 0.4039
Weighted F1     : 0.8948

MODEL: Balanced Random Forest

Training Time   : 6.77 seconds
Prediction Time : 0.99 seconds

Accuracy        : 0.8462
Macro Precision : 0.4138
Macro Recall    : 0.6543
Macro F1        : 0.4307
Weighted F1     : 0.8659


,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Training Time (s),Prediction Time (s)
0,XGBoost,0.908284,0.712953,0.486296,0.538460,0.905642,129.449561,1.635107
1,LightGBM,0.894658,0.487781,0.460968,0.461756,0.894130,87.237561,4.649495
2,Balanced Random Forest,0.846154,0.413796,0.654325,0.430703,0.865938,6.766607,0.987138
3,CatBoost,0.900117,0.566875,0.387150,0.403869,0.894848,295.935392,0.204439


In [47]:
# ============================================================
# COMPLETE DEVELOPMENT LEADERBOARD
# ============================================================

baseline_summary = dev_results_df[
    [
        "Model",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1",
        "Training Time (s)",
        "Prediction Time (s)"
    ]
].copy()

weighted_summary = weighted_results_df[
    [
        "Model",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1",
        "Training Time (s)",
        "Prediction Time (s)"
    ]
].copy()

smote_summary = smote_results_df[
    [
        "Model",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1",
        "Training Time (s)",
        "Prediction Time (s)"
    ]
].copy()

advanced_summary = advanced_results_df[
    [
        "Model",
        "Accuracy",
        "Macro Precision",
        "Macro Recall",
        "Macro F1",
        "Weighted F1",
        "Training Time (s)",
        "Prediction Time (s)"
    ]
].copy()

complete_leaderboard = pd.concat(
    [
        baseline_summary,
        weighted_summary,
        smote_summary,
        advanced_summary
    ],
    ignore_index=True
)

complete_leaderboard = complete_leaderboard.sort_values(
    by="Macro F1",
    ascending=False
).reset_index(drop=True)

complete_leaderboard

,Model,Accuracy,Macro Precision,Macro Recall,Macro F1,Weighted F1,Training Time (s),Prediction Time (s)
0,Random Forest (Balanced),0.904742,0.656933,0.508097,0.552423,0.902885,68.814152,1.602659
1,XGBoost,0.908284,0.712953,0.486296,0.538460,0.905642,129.449561,1.635107
2,Random Forest,0.903659,0.627959,0.498618,0.538330,0.901804,90.806255,1.097115
3,Decision Tree,0.881573,0.526958,0.506445,0.515201,0.881214,10.169917,0.051175
4,Random Forest (SMOTE),0.891074,0.505172,0.523031,0.510238,0.894772,1648.116062,1.387479
5,Decision Tree (Balanced),0.879073,0.509693,0.500691,0.504286,0.879218,6.468021,0.052161
6,LightGBM,0.894658,0.487781,0.460968,0.461756,0.894130,87.237561,4.649495
7,HistGradientBoosting,0.896366,0.504336,0.452758,0.453139,0.896364,14.658423,0.538044
8,Decision Tree (SMOTE),0.872281,0.439174,0.472484,0.450857,0.875964,154.188967,0.082578
9,Balanced Random Forest,0.846154,0.413796,0.654325,0.430703,0.865938,6.766607,0.987138


In [48]:
# ============================================================
# STEP 9.5 — PER-CLASS ANALYSIS
# ============================================================

for result in advanced_results:

    print("\n" + "=" * 70)
    print(result["Model"])
    print("=" * 70)

    print(
        classification_report(
            result["y_true"],
            result["y_pred"],
            zero_division=0
        )
    )


XGBoost
              precision    recall  f1-score   support

           0       0.99      0.98      0.98     17657
           1       0.33      0.10      0.16        39
           2       0.95      0.43      0.59        42
           3       0.70      0.18      0.28       318
           4       0.77      0.78      0.77      2409
           5       0.59      0.91      0.72      2074
           6       0.82      0.61      0.70       342
           7       0.81      0.53      0.64       944
           8       0.60      0.16      0.26       152
           9       0.57      0.19      0.29        21

    accuracy                           0.91     23998
   macro avg       0.71      0.49      0.54     23998
weighted avg       0.92      0.91      0.91     23998


LightGBM
              precision    recall  f1-score   support

           0       0.99      0.97      0.98     17657
           1       0.19      0.26      0.22        39
           2       0.27      0.38      0.32        42
     

In [49]:
CV_PIPELINES = {
    "Random Forest (Balanced)": WEIGHTED_PIPELINES["Random Forest (Balanced)"],
    "XGBoost": ADVANCED_PIPELINES["XGBoost"],
    "Random Forest": PIPELINES["Random Forest"],
    "Decision Tree": PIPELINES["Decision Tree"]
}

cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=SEED
)

In [50]:
def cross_validate_group_aware(
    model_name,
    pipeline,
    X_data,
    y_data,
    groups,
    cv
):
    fold_results = []

    print("\n" + "=" * 70)
    print(f"MODEL: {model_name}")
    print("=" * 70)

    for fold, (train_idx, val_idx) in enumerate(
        cv.split(X_data, y_data, groups=groups),
        start=1
    ):
        X_train_fold = X_data.iloc[train_idx]
        X_val_fold = X_data.iloc[val_idx]

        y_train_fold = y_data.iloc[train_idx]
        y_val_fold = y_data.iloc[val_idx]

        # Fresh unfitted pipeline for every fold
        fold_pipeline = clone(pipeline)

        start_train = time.perf_counter()

        fold_pipeline.fit(
            X_train_fold,
            y_train_fold
        )

        train_time = time.perf_counter() - start_train

        start_predict = time.perf_counter()

        y_pred = fold_pipeline.predict(X_val_fold)

        prediction_time = time.perf_counter() - start_predict

        accuracy = accuracy_score(
            y_val_fold,
            y_pred
        )

        macro_precision = precision_score(
            y_val_fold,
            y_pred,
            average="macro",
            zero_division=0
        )

        macro_recall = recall_score(
            y_val_fold,
            y_pred,
            average="macro",
            zero_division=0
        )

        macro_f1 = f1_score(
            y_val_fold,
            y_pred,
            average="macro",
            zero_division=0
        )

        weighted_f1 = f1_score(
            y_val_fold,
            y_pred,
            average="weighted",
            zero_division=0
        )

        fold_results.append({
            "Model": model_name,
            "Fold": fold,
            "Accuracy": accuracy,
            "Macro Precision": macro_precision,
            "Macro Recall": macro_recall,
            "Macro F1": macro_f1,
            "Weighted F1": weighted_f1,
            "Training Time (s)": train_time,
            "Prediction Time (s)": prediction_time
        })

        print(
            f"Fold {fold} | "
            f"Macro F1: {macro_f1:.4f} | "
            f"Macro Recall: {macro_recall:.4f} | "
            f"Accuracy: {accuracy:.4f}"
        )

    return pd.DataFrame(fold_results)

In [52]:
from sklearn.base import clone
cv_results_list = []

for model_name, pipeline in CV_PIPELINES.items():

    model_cv_results = cross_validate_group_aware(
        model_name=model_name,
        pipeline=pipeline,
        X_data=X_dev,
        y_data=y_dev,
        groups=groups_dev,
        cv=cv
    )

    cv_results_list.append(model_cv_results)

cv_results_df = pd.concat(
    cv_results_list,
    ignore_index=True
)


MODEL: Random Forest (Balanced)
Fold 1 | Macro F1: 0.5524 | Macro Recall: 0.5081 | Accuracy: 0.9047
Fold 2 | Macro F1: 0.5535 | Macro Recall: 0.5102 | Accuracy: 0.9038
Fold 3 | Macro F1: 0.5457 | Macro Recall: 0.5078 | Accuracy: 0.9023
Fold 4 | Macro F1: 0.5193 | Macro Recall: 0.4893 | Accuracy: 0.9010
Fold 5 | Macro F1: 0.5204 | Macro Recall: 0.4848 | Accuracy: 0.8985

MODEL: XGBoost
Fold 1 | Macro F1: 0.5385 | Macro Recall: 0.4863 | Accuracy: 0.9083
Fold 2 | Macro F1: 0.5307 | Macro Recall: 0.4828 | Accuracy: 0.9067
Fold 3 | Macro F1: 0.5331 | Macro Recall: 0.4971 | Accuracy: 0.9062
Fold 4 | Macro F1: 0.5182 | Macro Recall: 0.4820 | Accuracy: 0.9073
Fold 5 | Macro F1: 0.5162 | Macro Recall: 0.4768 | Accuracy: 0.9054

MODEL: Random Forest
Fold 1 | Macro F1: 0.5383 | Macro Recall: 0.4986 | Accuracy: 0.9037
Fold 2 | Macro F1: 0.5352 | Macro Recall: 0.4954 | Accuracy: 0.9031
Fold 3 | Macro F1: 0.5263 | Macro Recall: 0.4886 | Accuracy: 0.9017
Fold 4 | Macro F1: 0.5336 | Macro Recall: 0.5

In [54]:
CV_SUMMARY = (
    cv_results_df
    .groupby("Model")
    .agg(
        Accuracy_Mean=("Accuracy", "mean"),
        Accuracy_STD=("Accuracy", "std"),

        Macro_Precision_Mean=("Macro Precision", "mean"),
        Macro_Precision_STD=("Macro Precision", "std"),

        Macro_Recall_Mean=("Macro Recall", "mean"),
        Macro_Recall_STD=("Macro Recall", "std"),

        Macro_F1_Mean=("Macro F1", "mean"),
        Macro_F1_STD=("Macro F1", "std"),

        Weighted_F1_Mean=("Weighted F1", "mean"),
        Weighted_F1_STD=("Weighted F1", "std"),

        Training_Time_Mean=("Training Time (s)", "mean"),
        Prediction_Time_Mean=("Prediction Time (s)", "mean")
    )
    .sort_values(
        "Macro_F1_Mean",
        ascending=False
    )
    .reset_index()
)

CV_SUMMARY

,Model,Accuracy_Mean,Accuracy_STD,Macro_Precision_Mean,Macro_Precision_STD,Macro_Recall_Mean,Macro_Recall_STD,Macro_F1_Mean,Macro_F1_STD,Weighted_F1_Mean,Weighted_F1_STD,Training_Time_Mean,Prediction_Time_Mean
0,Random Forest (Balanced),0.902075,0.002453,0.631411,0.042589,0.500049,0.012011,0.538268,0.017088,0.900353,0.002363,60.337805,1.557388
1,XGBoost,0.906767,0.001094,0.668835,0.055933,0.484994,0.007574,0.527345,0.009690,0.904406,0.000889,128.544387,1.993004
2,Random Forest,0.902025,0.001714,0.612715,0.033143,0.488737,0.017225,0.524814,0.019626,0.900151,0.001761,65.726404,0.919343
3,Decision Tree,0.879892,0.001091,0.487108,0.023389,0.487400,0.018403,0.485856,0.019330,0.879814,0.000989,8.043280,0.046206
